In [6]:
import gym
import cv2
import time
import json
import random
import numpy as np
import pickle
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import os
from utils import DuelCNN, Agent
from collections import deque

ENVIRONMENT = 'ALE/Pong-v5'
DEVICE = 'cuda'
SAVE_MODELS = False  # Save models to file so you can test later
MODEL_PATH = "./models/pong-cnn-"  # Models path for saving or loading
SAVE_MODEL_INTERVAL = 10  # Save models at every X epoch
TRAIN_MODEL = False  # Train model while playing (Make it False when testing a model)
LOAD_MODEL_FROM_FILE = True  # Load model from file
LOAD_FILE_EPISODE = 900  # Load Xth episode from file
BATCH_SIZE = 64  # Minibatch size that select randomly from mem for train nets
MAX_EPISODE = 100000  # Max episode
MAX_STEP = 100000  # Max step size for one episode
NUM_EPISODES = 30
MAX_MEMORY_LEN = 50000  # Max memory len
MIN_MEMORY_LEN = 40000  # Min memory len before start train
from tqdm import tqdm
GAMMA = 0.97  # Discount rate
ALPHA = 0.00025  # Learning rate
EPSILON_DECAY = 0.99  # Epsilon decay rate by step
RENDER_GAME_WINDOW = False  # Opens a new window to render the game (Won't work on colab default)


if not os.path.exists('weights/'):
    os.mkdir('weights/')
if not os.path.exists('/export/kbodla/data/'):
    os.mkdir('/export/kbodla/data/')


In [2]:
import numpy as np
np.bool8 = bool  # Patch to use Python's `bool` type instead of NumPy's `bool8`

In [7]:
environment = gym.make(ENVIRONMENT,full_action_space=False) #, render_mode='human')  # Get env
agent = Agent(environment)  # Create Agent
if LOAD_MODEL_FROM_FILE:
    agent.online_model.load_state_dict(torch.load(MODEL_PATH+str(LOAD_FILE_EPISODE)+".pkl", map_location=torch.device('cpu')))
    with open(MODEL_PATH+str(LOAD_FILE_EPISODE)+'.json') as outfile:
        param = json.load(outfile)
        agent.epsilon = param.get('epsilon')
    startEpisode = LOAD_FILE_EPISODE + 1
else:
    startEpisode = 1

last_100_ep_reward = deque(maxlen=100)  # Last 100 episode rewards
total_step = 1  # Cumulkative sum of all steps in episodes

# Collecting Data for experiments
all_rewards = list()
all_states = list()
all_x = list()
all_actions = list()

for episode in tqdm(range(startEpisode, startEpisode + NUM_EPISODES)):
    startTime = time.time()  # Keep time
    state,info = environment.reset()  # Reset env
    # all_states.append(state.tolist())
    state = agent.preProcess(state)  # Process image
    # Stack state . Every state contains 4 time contionusly frames
    # We stack frames like 4 channel image
    state = np.stack((state, state, state, state))
    # all_states.append(state)
    total_max_q_val = 0  # Total max q vals
    total_reward = 0  # Total reward for each episode
    total_loss = 0  # Total loss for each episode

    for step in range(MAX_STEP):
        # Select and perform an action
        action, latent_x = agent.act(state)  # Act

        #### Save x and actions for training wrapper model 
        # all_x.append(latent_x.tolist()[0])
        all_x.append(latent_x.detach().cpu().numpy())
        all_actions.append(action)
        all_states.append(torch.tensor(state))
        next_state, reward, done, _,info = environment.step(action)  # Observe

        # #### Save State for human-defined concepts -- uncomment if you want to manually save the observations to select prototypes later
        # temp = torch.tensor(next_state).permute(2, 0, 1).view(1, 3, 210, 160)
        # temp = torch.nn.functional.interpolate(temp, scale_factor=0.1, mode='nearest')[0].permute(1, 2, 0)
        # temp = temp.tolist()            
        # all_states.append(temp)

        next_state = agent.preProcess(next_state)  # Process image

        # Stack state . Every state contains 4 time contionusly frames
        # We stack frames like 4 channel image
        next_state = np.stack((next_state, state[0], state[1], state[2]))

        # Store the transition in memory
        agent.storeResults(state, action, reward, next_state, done)  # Store to mem

        # Move to the next state
        state = next_state  # Update state

        if TRAIN_MODEL:
            # Perform one step of the optimization (on the target network)
            loss, max_q_val = agent.train()  # Train with random BATCH_SIZE state taken from mem
        else:
            loss, max_q_val = [0, 0]

        total_loss += loss
        total_max_q_val += max_q_val
        total_reward += reward
        total_step += 1
        if total_step % 1000 == 0:
            agent.adaptiveEpsilon()  # Decrase epsilon

        if done:  # Episode completed
            currentTime = time.time()  # Keep current time
            time_passed = currentTime - startTime  # Find episode duration
            current_time_format = time.strftime("%H:%M:%S", time.gmtime())  # Get current dateTime as HH:MM:SS
            epsilonDict = {'epsilon': agent.epsilon}  # Create epsilon dict to save model as file

            if SAVE_MODELS and episode % SAVE_MODEL_INTERVAL == 0:  # Save model as file
                weightsPath = MODEL_PATH + str(episode) + '.pkl'
                epsilonPath = MODEL_PATH + str(episode) + '.json'

                torch.save(agent.online_model.state_dict(), weightsPath)
                with open(epsilonPath, 'w') as outfile:
                    json.dump(epsilonDict, outfile)

            if TRAIN_MODEL:
                agent.target_model.load_state_dict(agent.online_model.state_dict())  # Update target model

            last_100_ep_reward.append(total_reward)
            avg_max_q_val = total_max_q_val / step

            outStr = "Episode:{} Time:{} Reward:{:.2f} Loss:{:.2f} Last_100_Avg_Rew:{:.3f} Avg_Max_Q:{:.3f} Epsilon:{:.2f} Duration:{:.2f} Step:{} CStep:{}".format(
                episode, current_time_format, total_reward, total_loss, np.mean(last_100_ep_reward), avg_max_q_val, agent.epsilon, time_passed, step, total_step
            )
            print(step)
            print(outStr)

            all_rewards.append(total_reward)

            if SAVE_MODELS:
                outputPath = MODEL_PATH + "out" + '.txt'  # Save outStr to file
                with open(outputPath, 'a') as outfile:
                    outfile.write(outStr+"\n")

            break

print("Average Reward:", sum(all_rewards) / NUM_EPISODES)

NamespaceNotFound: Namespace ALE not found. Have you installed the proper package for ALE?

In [4]:
# We're not saving the observations in the interest of saving memory, but you can uncomment if you like.
with open('/export/kbodla/data/X_train.pkl', 'wb') as f:
  # pickle.dump(all_states, f)
    pickle.dump(all_x,f)
with open('/export/kbodla/data/a_train.pkl', 'wb') as f:
  pickle.dump(all_actions, f)
with open('/export/kbodla/data/obs_train.pkl', 'wb') as f:
  pickle.dump(all_states, f)